# Pixels to Predictions: CS-GY 6953 (Deep Learning) Kaggle Competition
### Author: Mariia Onokhina
**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

## Installation of Libraries 

Before installing any libraries, I create a new Conda environment and add it to Jupyter Notebook to ensure I start from a clean slate and that the code is reproducible.

**Run the following code in a terminal if you'd like to start fresh with a new environment:**
```bash
conda create -n pixels-to-predictions python=3.10
conda activate pixels-to-predictions
conda install -c conda-forge notebook
conda install -c conda-forge ipykernel
python -m ipykernel install --user --name pixels-to-predictions --display-name "Pixels-to-predictions DL"
```

IMPORTANT: Manually change the Kernel in Jupyter Notebook in VS Code or Jupyter Lab to "Pixels-to-predictions DL".

In [3]:
# Uncomment this cell to install the necessary Python packages.
import sys
print("Python:", sys.executable)
!{sys.executable} -m pip install -q transformers==4.57.6 peft==0.18.1 kaggle matplotlib scikit-learn pandas numpy ipywidgets jupyterlab_widgets bitsandbytes accelerate datasets pillow --quiet

Python: /home/devvingduo/miniforge3/envs/pixels-to-predictions/bin/python


---
## Imports & Configuration

In [24]:
# Imports & Configuration
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

import ast

In [5]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
# Paths 
DATA_DIR = Path("data")

# Model
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# Basic Settings
IMG_SIZE = 224

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3090


---
### Load and Preprocess Data

Download data from https://www.kaggle.com/competitions/pixels-to-predictions/data via Kaggle CLI.

For this, you need a Legacy API key which you can create here: https://www.kaggle.com/settings.

When you create a new key, it will download a ```kaggle.json```. 

In your terminal, run:
```bash
mkdir -p ~/.kaggle
mv <your_downloads_folder> ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json
```

Verify that it worked by running: 
```bash
ls -la ~/.kaggle
```

I have to add it manually because I'm working via SSH into my Linux server machine.

In [ ]:
# Uncomment this cell to download the data in a .zip file
#!kaggle competitions download -c pixels-to-predictions

 83%|█████████████████████████████████▍      | 299M/358M [00:00<00:00, 3.13GB/s]
100%|████████████████████████████████████████| 358M/358M [00:00<00:00, 3.15GB/s]


In [ ]:
# Uncomment this cell to unzip the data into "data" folder
#!unzip pixels-to-predictions.zip -d data

Archive:  pixels-to-predictions.zip
  inflating: data/images/images/test/test_00001.png  
  inflating: data/images/images/test/test_00002.png  
  inflating: data/images/images/test/test_00005.png  
  inflating: data/images/images/test/test_00009.png  
  inflating: data/images/images/test/test_00013.png  
  inflating: data/images/images/test/test_00016.png  
  inflating: data/images/images/test/test_00017.png  
  inflating: data/images/images/test/test_00018.png  
  inflating: data/images/images/test/test_00021.png  
  inflating: data/images/images/test/test_00023.png  
  inflating: data/images/images/test/test_00034.png  
  inflating: data/images/images/test/test_00042.png  
  inflating: data/images/images/test/test_00050.png  
  inflating: data/images/images/test/test_00055.png  
  inflating: data/images/images/test/test_00064.png  
  inflating: data/images/images/test/test_00067.png  
  inflating: data/images/images/test/test_00069.png  
  inflating: data/images/images/test/test_0007

In [13]:
# Load CSVs
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

Inspect the data. 

In [20]:
train_df.head(2)

,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,"[""the male's tadpoles will be larger when they...",3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[""the female's offspring will live longer"", ""t...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [15]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3109 entries, 0 to 3108
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           3109 non-null   object
 1   image_path   3109 non-null   object
 2   question     3109 non-null   object
 3   choices      3109 non-null   object
 4   num_choices  3109 non-null   int64 
 5   answer       3109 non-null   int64 
 6   hint         2385 non-null   object
 7   lecture      2669 non-null   object
 8   solution     2580 non-null   object
 9   task         3109 non-null   object
 10  grade        3109 non-null   object
 11  subject      3109 non-null   object
 12  topic        3109 non-null   object
 13  category     3109 non-null   object
 14  skill        3109 non-null   object
dtypes: int64(2), object(13)
memory usage: 364.5+ KB


In [21]:
val_df.head(2)

,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,val_00671,images/val/val_00671.png,Why might covering its eggs with its body incr...,"[""the leech's eggs will hatch"", ""the leech wil...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,val_04111,images/val/val_04111.png,Why might fanning eggs increase the reproducti...,"[""the male will build a nest for females to la...",3,1,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [18]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048 entries, 0 to 1047
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           1048 non-null   object
 1   image_path   1048 non-null   object
 2   question     1048 non-null   object
 3   choices      1048 non-null   object
 4   num_choices  1048 non-null   int64 
 5   answer       1048 non-null   int64 
 6   hint         816 non-null    object
 7   lecture      915 non-null    object
 8   solution     876 non-null    object
 9   task         1048 non-null   object
 10  grade        1048 non-null   object
 11  subject      1048 non-null   object
 12  topic        1048 non-null   object
 13  category     1048 non-null   object
 14  skill        1048 non-null   object
dtypes: int64(2), object(13)
memory usage: 122.9+ KB


In [22]:
test_df.head(2)

,id,image_path,question,choices,num_choices,hint,lecture,task,grade,subject,topic,category,skill
0,test_01750,images/test/test_01750.png,"Based on clues in the text, why would farmers ...","[""The cats were thought to be visiting goddess...",4,Read the text about cats.\nCats are among the ...,"Informational texts include many facts, exampl...",closed choice,grade5,language science,reading-comprehension,Informational texts: level 1,Read passages about animals
1,test_00128,images/test/test_00128.png,What is the probability that an American curl ...,"[""0/4"", ""2/4"", ""4/4"", ""3/4"", ""1/4""]",5,"In a group of American curl cats, some individ...",Offspring genotypes: homozygous or heterozygou...,closed choice,grade8,natural science,biology,Genes to traits,Use Punnett squares to calculate probabilities...


In [23]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1008 entries, 0 to 1007
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           1008 non-null   object
 1   image_path   1008 non-null   object
 2   question     1008 non-null   object
 3   choices      1008 non-null   object
 4   num_choices  1008 non-null   int64 
 5   hint         794 non-null    object
 6   lecture      825 non-null    object
 7   task         1008 non-null   object
 8   grade        1008 non-null   object
 9   subject      1008 non-null   object
 10  topic        1008 non-null   object
 11  category     1008 non-null   object
 12  skill        1008 non-null   object
dtypes: int64(1), object(12)
memory usage: 102.5+ KB


It seems like most of the columns are non-null, besides ```lecture``` and ```hint```.

In [25]:
# Function to parse the choices column using ast module (Abstract Syntax Tree)
def parse_choices(x):
    # If x is already a list, return it
    if isinstance(x, list):
        return x
    # If x is a string, parse it using ast.literal_eval
    return ast.literal_eval(x)

The ```choices``` column is a JSON string, so we parse it using the function above.

In [26]:
for df in [train_df, val_df, test_df]:
    df["choices_list"] = df["choices"].apply(parse_choices)

In [28]:
train_df[["id", "question", "choices_list", "answer"]].head(2)

,id,question,choices_list,answer
0,train_07667,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,2
1,train_02628,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",0


In [29]:
train_df["choices_list"].iloc[0]

["the male's tadpoles will be larger when they hatch",
 'the male will carry his tadpoles through the forest',
 "the male's tadpoles will become adult frogs"]

Check lengths of the dataframes.

In [31]:
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

Train: 3,109 | Val: 1,048 | Test: 1,008


Check answer validity. If correct answer index is less than 0 or bigger than the number of choices, it's invalid and we should not train on them.

In [35]:
def check_answers(df, name):
    bad = df[(df["answer"] < 0) | (df["answer"] >= df["num_choices"])]
    print(name, "bad answers:", len(bad))

check_answers(train_df, "train")
check_answers(val_df, "val")

train bad answers: 0
val bad answers: 0


There are no invalid answers.

Check class distribution, whether some answers repeat more often that others. 

In [36]:
print(train_df["answer"].value_counts(normalize=True).sort_index())
print(val_df["answer"].value_counts(normalize=True).sort_index())

answer
0    0.361531
1    0.330653
2    0.237054
3    0.065616
4    0.005146
Name: proportion, dtype: float64
answer
0    0.331107
1    0.354962
2    0.235687
3    0.071565
4    0.006679
Name: proportion, dtype: float64


There is a class imbalance. There is a strong skew toward: 0 and 1 (~70%). Guessing 0 and 1 would be correct most of the time, while guessing 3 or 4 is rare.

---
### Prompt Engineering

In [4]:
# ── 2b. Prompt Engineering ───────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    """
    Builds the text prompt for the Vision Language Model.
    The <image> token is required for the model to process the image.
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

# Display an example prompt
print(build_prompt(train_df.iloc[0], include_answer=True))

<image>
Context:
Maps have four cardinal directions, or main directions. Those directions are north, south, east, and west.
A compass rose is a set of arrows that point to the cardinal directions. A compass rose usually shows only the first letter of each cardinal direction.
The north arrow points to the North Pole. On most maps, north is at the top of the map.

Question: Which of these states is farthest north?
Choices:
  A. West Virginia
  B. Louisiana
  C. Arizona
  D. Oklahoma
Answer: A


In [5]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img = Image.open(self.data_dir / rel_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }

train_ds = ScienceQADataset(train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

Datasets created: train=6218, val=2097, test=2017


## 3. Model Loading and Inference Example

This section loads `HuggingFaceTB/SmolVLM-500M-Instruct` and runs a quick inference example on one validation sample.

In [6]:
# ── 3a. Load SmolVLM model + run one inference example ───────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
 )
if not torch.cuda.is_available():
    model.to(device)
model.eval()

# Pick a sample from validation set
sample = val_df.iloc[0]
sample_image = Image.open(DATA_DIR / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(
    text=[sample_prompt],
    images=[sample_image],
    return_tensors="pt",
)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )

decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Prompt:")
print(sample_prompt)
print("\nModel output:")
print(decoded)
print(f"\nGround-truth answer index: {sample['answer']}")

/root/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/root/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/.venv/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Prompt:
<image>
Context:
An adaptation is an inherited trait that helps an organism survive or reproduce. Adaptations can include both body parts and behaviors.
The shape of an animal's mouth is one example of an adaptation. Animals' mouths can be adapted in different ways. For example, a large mouth with sharp teeth might help an animal tear through meat. A long, thin mouth might help an animal catch insects that live in holes. Animals that eat similar food often have similar mouths.
Sturgeons eat invertebrates, plants, and small fish. They are bottom feeders. Bottom feeders find their food at the bottom of rivers, lakes, and the ocean.
The 's mouth is located on the underside of its head and points downward. Its mouth is adapted for bottom feeding.
Figure: sturgeon.

Question: Which animal's mouth is also adapted for bottom feeding?
Choices:
  A. discus
  B. armored catfish
Answer:

Model output:






Context:
An adaptation is an inherited trait that helps an organism survive or rep